# Module 8 — Deep Learning for Computational Materials
## Hands-On Python / Jupyter Tutorial

**Level:** IIT M.Tech / PhD Applied Materials / Computational Materials  
**Prerequisites:** Modules 1–7  
**Suggested duration:** 4–5 tutorials × 3 hours

### Module philosophy

Deep learning is introduced only after students understand numerical methods, linear algebra, data analysis, feature engineering, and classical machine learning.

The goal is not to teach neural-network syntax. Students should understand:

**materials problem → representation → neural network → loss → gradients → optimization → validation → physical interpretation**

The examples use synthetic materials datasets so that the notebook runs without external databases.


# Learning objectives

Students should be able to:

1. Explain why neural networks represent nonlinear functions.
2. Derive a single neuron and its gradient.
3. Understand forward propagation and backpropagation.
4. Train multilayer perceptrons (MLPs).
5. Use training, validation, and test datasets correctly.
6. Apply scaling, regularization, dropout, and early stopping.
7. Build regression and classification networks.
8. Diagnose overfitting and underfitting.
9. Compare deep learning with classical ML.
10. Understand CNNs for microstructure/images.
11. Understand the motivation for autoencoders and graph neural networks.
12. Design a scientifically valid materials deep-learning workflow.


# 1. Environment

We use NumPy, Pandas, Matplotlib, scikit-learn, and PyTorch.

The notebook is intended for Jupyter Notebook, JupyterLab, or Google Colab.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)


# 2. Neural-network mathematics

A neuron computes

`z = w^T x + b`

followed by an activation

`a = phi(z)`.

A multilayer network is a composition of such transformations:

`x -> layer 1 -> layer 2 -> ... -> prediction`.

For regression, a common loss is mean squared error:

`MSE = mean((y - y_hat)^2)`.

Training updates parameters according to gradient descent:

`theta_new = theta_old - learning_rate * gradient(loss)`.

Backpropagation is repeated application of the chain rule.


# 3. Materials dataset

We construct synthetic descriptors resembling quantities used in materials science:

- atomic-size mismatch
- mean electronegativity
- density
- grain size
- processing temperature
- cooling rate
- defect fraction
- elastic modulus

The target is a synthetic material property with nonlinear interactions.


In [ ]:
rng = np.random.default_rng(42)
n = 3000

df = pd.DataFrame({
    "atomic_size_mismatch": rng.uniform(0.0, 0.18, n),
    "mean_electronegativity": rng.normal(1.8, 0.35, n),
    "density": rng.normal(7.0, 0.9, n),
    "grain_size_um": rng.lognormal(np.log(12), 0.55, n),
    "processing_temperature_K": rng.normal(1100, 160, n),
    "cooling_rate_K_s": rng.lognormal(np.log(20), 0.8, n),
    "defect_fraction": np.clip(
        rng.lognormal(np.log(0.01), 0.7, n), 0, 0.12
    ),
    "elastic_modulus_GPa": rng.normal(180, 35, n)
})

df["target_property"] = (
    150
    + 0.75 * df["elastic_modulus_GPa"]
    + 90 / np.sqrt(df["grain_size_um"])
    - 220 * df["atomic_size_mismatch"]**2
    + 55 * np.sin(df["mean_electronegativity"])
    + 0.08 * df["processing_temperature_K"]
    - 18 * np.log1p(df["cooling_rate_K_s"])
    - 900 * df["defect_fraction"]
    + 160 * df["atomic_size_mismatch"]
      * np.sin(df["processing_temperature_K"] / 180)
    + rng.normal(0, 30, n)
)

df.head()


# 4. Exploratory analysis


In [ ]:
display(df.describe().T)

plt.figure(figsize=(8, 5))
plt.hist(df["target_property"], bins=40)
plt.xlabel("Target property")
plt.ylabel("Count")
plt.title("Target distribution")
plt.grid(alpha=0.25)
plt.show()


# 5. Train / validation / test split

Use three conceptual datasets:

- **Training:** optimize neural-network parameters.
- **Validation:** choose architecture and hyperparameters.
- **Test:** final unbiased evaluation.

The test set should not guide model development.


In [ ]:
X = df.drop(columns=["target_property"]).to_numpy(dtype=np.float32)
y = df["target_property"].to_numpy(dtype=np.float32).reshape(-1, 1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)


# 6. Feature scaling

Standardization is

`z = (x - mean) / standard_deviation`.

The mean and standard deviation must be calculated from **training data only**.


In [ ]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Training means:", X_train_s.mean(axis=0).round(4))
print("Training std:", X_train_s.std(axis=0).round(4))


# 7. Convert to PyTorch datasets


In [ ]:
Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32)

Xv = torch.tensor(X_val_s, dtype=torch.float32)
yv = torch.tensor(y_val, dtype=torch.float32)

Xte = torch.tensor(X_test_s, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(Xv, yv),
    batch_size=256
)

test_loader = DataLoader(
    TensorDataset(Xte, yte),
    batch_size=256
)


# 8. Single neuron

A single linear neuron is

`y_hat = w^T x + b`.

This is mathematically equivalent to a linear regression model.

This connection is important: neural networks extend familiar linear algebra rather than replacing it.


In [ ]:
class SingleNeuron(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

single = SingleNeuron(X_train_s.shape[1]).to(device)
loss_fn = nn.MSELoss()

optimizer = torch.optim.SGD(
    single.parameters(),
    lr=0.01
)

for epoch in range(150):
    optimizer.zero_grad()

    prediction = single(Xtr.to(device))
    loss = loss_fn(prediction, ytr.to(device))

    loss.backward()
    optimizer.step()

print("Final training loss:", loss.item())


# 9. Gradient calculation by hand

For

`z = w*x + b`

`y_hat = z`

`L = 0.5 * (y_hat - y)^2`

the derivatives are

`dL/dy_hat = y_hat - y`

`dL/dw = (y_hat - y) * x`

`dL/db = y_hat - y`.

This is the simplest example of backpropagation.


In [ ]:
x = 2.0
y_true = 5.0
w = 1.5
b = 0.5

y_hat = w*x + b
loss = 0.5 * (y_hat - y_true)**2

dL_dyhat = y_hat - y_true
dL_dw = dL_dyhat * x
dL_db = dL_dyhat

print("prediction =", y_hat)
print("loss =", loss)
print("dL/dw =", dL_dw)
print("dL/db =", dL_db)


# Exercise 1 — Connect neural networks to linear regression

1. Train the single neuron with several learning rates.
2. Compare it with `sklearn.LinearRegression`.
3. Explain why removing all nonlinear activations makes a multilayer network equivalent to a linear transformation.


# 10. Activation functions

Common activations include:

- ReLU: `max(0, z)`
- sigmoid: `1 / (1 + exp(-z))`
- tanh: `tanh(z)`

Nonlinearity is essential for a multilayer network to represent nonlinear materials-property relationships.


In [ ]:
z = np.linspace(-6, 6, 500)

relu = np.maximum(0, z)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)

plt.figure(figsize=(8, 5))
plt.plot(z, relu, label="ReLU")
plt.plot(z, sigmoid, label="Sigmoid")
plt.plot(z, tanh, label="Tanh")
plt.xlabel("z")
plt.ylabel("Activation")
plt.title("Activation functions")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 11. Multilayer perceptron

A two-hidden-layer network can be written as

`h1 = ReLU(W1*x + b1)`

`h2 = ReLU(W2*h1 + b2)`

`y_hat = W3*h2 + b3`.

The hidden layers allow nonlinear feature interactions.


In [ ]:
class MaterialsMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

model = MaterialsMLP(X_train_s.shape[1]).to(device)
print(model)


# 12. Reusable training and evaluation functions


In [ ]:
def evaluate_regression(model, loader):
    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(device)).cpu().numpy()
            predictions.append(pred)
            targets.append(yb.numpy())

    predictions = np.vstack(predictions).ravel()
    targets = np.vstack(targets).ravel()

    return {
        "MAE": mean_absolute_error(targets, predictions),
        "RMSE": np.sqrt(mean_squared_error(targets, predictions)),
        "R2": r2_score(targets, predictions)
    }


def train_model(model, train_loader, val_loader,
                epochs=200, lr=1e-3, weight_decay=0.0):

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        total = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            total += loss.item() * len(xb)

        train_loss = total / len(train_loader.dataset)

        model.eval()
        total = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = criterion(pred, yb)
                total += loss.item() * len(xb)

        val_loss = total / len(val_loader.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

    return history


# 13. Train the MLP


In [ ]:
history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=250,
    lr=1e-3
)

print(evaluate_regression(model, test_loader))


# 14. Learning curves

Interpret the gap between training and validation loss.

Typical patterns:

- both high → underfitting
- training low, validation high → overfitting
- both decrease and stabilize → useful learning


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Training")
plt.plot(history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("MLP learning curves")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 15. Predicted versus true values


In [ ]:
model.eval()

with torch.no_grad():
    pred = model(Xte.to(device)).cpu().numpy().ravel()

plt.figure(figsize=(7, 7))
plt.scatter(y_test.ravel(), pred, alpha=0.55)

lo = min(y_test.min(), pred.min())
hi = max(y_test.max(), pred.max())

plt.plot([lo, hi], [lo, hi], "--")
plt.xlabel("True property")
plt.ylabel("Predicted property")
plt.title("MLP regression")
plt.grid(alpha=0.25)
plt.show()

print("MAE:", mean_absolute_error(y_test.ravel(), pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test.ravel(), pred)))
print("R2:", r2_score(y_test.ravel(), pred))


# 16. Residual analysis

Residual:

`e = y_true - y_pred`.

Look for:

- systematic curvature
- changing variance
- outliers
- regions where the model fails

A single RMSE number cannot reveal these patterns.


In [ ]:
residual = y_test.ravel() - pred

plt.figure(figsize=(8, 5))
plt.scatter(pred, residual, alpha=0.55)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted property")
plt.ylabel("Residual")
plt.title("MLP residuals")
plt.grid(alpha=0.25)
plt.show()


# 17. Regularization

Common strategies:

- weight decay / L2 regularization
- dropout
- early stopping
- simpler architectures
- more data

Weight decay penalizes large weights and can improve generalization.


In [ ]:
regularized_model = MaterialsMLP(
    X_train_s.shape[1]
).to(device)

regularized_history = train_model(
    regularized_model,
    train_loader,
    val_loader,
    epochs=250,
    lr=1e-3,
    weight_decay=1e-4
)

print(evaluate_regression(
    regularized_model,
    test_loader
))


# 18. Dropout

Dropout randomly removes a fraction of hidden activations during training.

It is a regularization method, not a replacement for good data splitting.

PyTorch uses:

- `model.train()` → dropout active
- `model.eval()` → dropout inactive


In [ ]:
class DropoutMLP(nn.Module):
    def __init__(self, n_features, p=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

dropout_model = DropoutMLP(
    X_train_s.shape[1], p=0.20
).to(device)

dropout_history = train_model(
    dropout_model,
    train_loader,
    val_loader,
    epochs=250,
    lr=1e-3
)

print(evaluate_regression(
    dropout_model,
    test_loader
))


# 19. Early stopping

If validation loss reaches a minimum and subsequently rises, continued training may overfit.

Early stopping:

1. monitor validation loss
2. save the best model
3. stop after no improvement for a chosen patience period
4. restore the best state


In [ ]:
def train_with_early_stopping(
    model, train_loader, val_loader,
    epochs=500, lr=1e-3, patience=30
):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val = np.inf
    best_state = None
    wait = 0

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        total = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

            total += loss.item() * len(xb)

        train_loss = total / len(train_loader.dataset)

        model.eval()
        total = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                total += criterion(
                    model(xb), yb
                ).item() * len(xb)

        val_loss = total / len(val_loader.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print("Stopped at epoch:", epoch + 1)
            break

    model.load_state_dict(best_state)
    return history

early_model = MaterialsMLP(
    X_train_s.shape[1]
).to(device)

early_history = train_with_early_stopping(
    early_model,
    train_loader,
    val_loader,
    epochs=500,
    lr=1e-3,
    patience=30
)

print(evaluate_regression(
    early_model,
    test_loader
))


# 20. Learning rate experiment

The learning rate controls the size of parameter updates.

Compare:

- `1e-4`
- `1e-3`
- `1e-2`

Too small can be slow; too large can make training unstable.


In [ ]:
learning_rates = [1e-4, 1e-3, 1e-2]
lr_histories = {}

for lr in learning_rates:
    test_model = MaterialsMLP(
        X_train_s.shape[1]
    ).to(device)

    lr_histories[lr] = train_model(
        test_model,
        train_loader,
        val_loader,
        epochs=150,
        lr=lr
    )

plt.figure(figsize=(8, 5))

for lr, hist in lr_histories.items():
    plt.plot(hist["val_loss"], label=f"lr={lr}")

plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Learning-rate comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Exercise 2 — Hyperparameter study

Change one variable at a time:

- learning rate
- hidden-layer width
- number of layers
- batch size
- dropout
- weight decay

Record:

- best validation RMSE
- epoch of best validation RMSE
- final test RMSE

Do not select a model using test performance.


# 21. Classification

For binary classification, the network produces a logit `z`.

Probability:

`p = sigmoid(z)`.

Binary cross-entropy is used as the training objective. PyTorch's `BCEWithLogitsLoss` combines the sigmoid and loss calculation in a numerically stable implementation.


In [ ]:
y_class = (
    df["target_property"]
    > df["target_property"].median()
).astype(np.float32).to_numpy().reshape(-1, 1)

Xc = df.drop(columns=["target_property"]).to_numpy(
    dtype=np.float32
)

Xc_train, Xc_temp, yc_train, yc_temp = train_test_split(
    Xc, y_class, test_size=0.30,
    random_state=42, stratify=y_class
)

Xc_val, Xc_test, yc_val, yc_test = train_test_split(
    Xc_temp, yc_temp, test_size=0.50,
    random_state=42, stratify=yc_temp
)

cls_scaler = StandardScaler()

Xc_train = cls_scaler.fit_transform(Xc_train)
Xc_val = cls_scaler.transform(Xc_val)
Xc_test = cls_scaler.transform(Xc_test)

Xc_train = torch.tensor(Xc_train, dtype=torch.float32)
Xc_val = torch.tensor(Xc_val, dtype=torch.float32)
Xc_test = torch.tensor(Xc_test, dtype=torch.float32)

yc_train = torch.tensor(yc_train, dtype=torch.float32)
yc_val = torch.tensor(yc_val, dtype=torch.float32)
yc_test = torch.tensor(yc_test, dtype=torch.float32)

cls_train_loader = DataLoader(
    TensorDataset(Xc_train, yc_train),
    batch_size=64, shuffle=True
)

cls_test_loader = DataLoader(
    TensorDataset(Xc_test, yc_test),
    batch_size=256
)


In [ ]:
class MaterialsClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

classifier = MaterialsClassifier(
    Xc_train.shape[1]
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    classifier.parameters(), lr=1e-3
)

for epoch in range(150):
    classifier.train()

    for xb, yb in cls_train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = classifier(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

classifier.eval()

with torch.no_grad():
    logits = classifier(
        Xc_test.to(device)
    ).cpu().numpy().ravel()

probabilities = 1 / (1 + np.exp(-logits))
classes = (probabilities >= 0.5).astype(int)

print("Accuracy:",
      accuracy_score(yc_test.numpy().ravel(), classes))


# 22. Confusion matrix


In [ ]:
cm = confusion_matrix(
    yc_test.numpy().ravel(),
    classes
)

ConfusionMatrixDisplay(
    confusion_matrix=cm
).plot()

plt.title("Neural-network classification")
plt.show()


# 23. CNNs for materials microstructures

Images and spatial fields are natural inputs for convolutional neural networks.

Potential materials applications:

- SEM/TEM image classification
- phase maps
- porosity maps
- grain structures
- diffraction images
- spatial simulation fields

A convolution applies a local kernel across the image. Parameter sharing makes CNNs efficient at recognizing spatial patterns.


# 24. Synthetic microstructure dataset

We create binary images containing circular inclusions.

The target is a synthetic property related to inclusion fraction.


In [ ]:
def make_microstructure(size=32, n_circles=8):
    image = np.zeros((size, size), dtype=np.float32)

    yy, xx = np.mgrid[:size, :size]

    for _ in range(n_circles):
        cx = rng.integers(0, size)
        cy = rng.integers(0, size)
        radius = rng.integers(2, 6)

        mask = (xx-cx)**2 + (yy-cy)**2 <= radius**2
        image[mask] = 1.0

    return image

images = np.array([
    make_microstructure()
    for _ in range(1000)
])

volume_fraction = images.mean(axis=(1, 2))

micro_target = (
    200 + 500 * volume_fraction
    + rng.normal(0, 15, len(images))
)

print(images.shape)


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, image in zip(axes, images[:5]):
    ax.imshow(image)
    ax.axis("off")

plt.suptitle("Synthetic microstructures")
plt.show()


# 25. CNN regression


In [ ]:
Ximg_train, Ximg_test, yimg_train, yimg_test = train_test_split(
    images, micro_target,
    test_size=0.20,
    random_state=42
)

Ximg_train = torch.tensor(
    Ximg_train[:, None, :, :],
    dtype=torch.float32
)

Ximg_test = torch.tensor(
    Ximg_test[:, None, :, :],
    dtype=torch.float32
)

yimg_train = torch.tensor(
    yimg_train.reshape(-1, 1),
    dtype=torch.float32
)

yimg_test = torch.tensor(
    yimg_test.reshape(-1, 1),
    dtype=torch.float32
)

img_train_loader = DataLoader(
    TensorDataset(Ximg_train, yimg_train),
    batch_size=32,
    shuffle=True
)

img_test_loader = DataLoader(
    TensorDataset(Ximg_test, yimg_test),
    batch_size=64
)


In [ ]:
class MicrostructureCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.regressor(self.features(x))

cnn = MicrostructureCNN().to(device)
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    cnn.parameters(), lr=1e-3
)

for epoch in range(80):
    cnn.train()

    for xb, yb in img_train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(cnn(xb), yb)
        loss.backward()
        optimizer.step()

cnn.eval()
predictions = []

with torch.no_grad():
    for xb, yb in img_test_loader:
        predictions.append(
            cnn(xb.to(device)).cpu().numpy()
        )

cnn_pred = np.vstack(predictions).ravel()

print("CNN MAE:",
      mean_absolute_error(yimg_test.numpy().ravel(), cnn_pred))
print("CNN RMSE:",
      np.sqrt(mean_squared_error(yimg_test.numpy().ravel(), cnn_pred)))
print("CNN R2:",
      r2_score(yimg_test.numpy().ravel(), cnn_pred))


# 26. Autoencoders

An autoencoder learns:

`input -> latent representation -> reconstruction`.

The latent vector can be much smaller than the input.

Potential materials applications:

- microstructure compression
- anomaly detection
- latent-space visualization
- reduced-order modelling
- representation learning


# 27. Graph neural networks

Many materials are naturally graphs:

- atoms = nodes
- atomic neighbors/bonds = edges

A simplified message-passing step is:

`h_i(new) = update(h_i, aggregate(messages from neighbors))`.

Graph neural networks are attractive for crystal and molecular structures because they can work directly with relational structure instead of requiring fixed handcrafted descriptors.

Students should investigate PyTorch Geometric, DGL, matgl, MEGNet-family models, CHGNet-family approaches, and related atomistic ML frameworks after mastering this module.


# 28. Scientific validation and data leakage

Deep learning does not solve poor experimental design.

Potential leakage includes:

- scaling using the complete dataset
- choosing epochs from test performance
- duplicate materials across splits
- structures from the same simulation trajectory in different splits
- multiple measurements from one sample appearing in train and test
- related compositions appearing in both sets when the intended task is extrapolation

The data split should reproduce the **real prediction scenario**.


# 29. Domain of applicability

A neural network may interpolate well while extrapolating poorly.

Ask:

- Are the new materials inside the training domain?
- Are the compositions similar?
- Are processing conditions similar?
- Are the microstructures represented?
- Are the labels noisy?
- Does the model violate known physical constraints?

A high R² is not proof of physical validity.


# 30. Physics-informed thinking

Future scientific-ML methods may impose:

- conservation laws
- symmetry
- boundary conditions
- differential equations
- thermodynamic constraints
- rotational/equivariant structure

This leads naturally toward:

- Physics-Informed Neural Networks (PINNs)
- neural operators
- equivariant neural networks
- differentiable simulation
- learned constitutive models
- ML surrogate models for PDEs


# 31. Neural surrogate modelling

A powerful computational-materials workflow is:

**expensive numerical simulation → simulation database → neural surrogate**

For example:

`(D, T, t, boundary conditions) -> concentration field`

or

`(composition, temperature, strain rate) -> stress-strain response`.

Potential benefits:

- rapid parameter sweeps
- optimization
- inverse problems
- process control
- uncertainty studies
- materials screening


# 32. Materials deep-learning ecosystem

### General
- PyTorch
- TensorFlow / Keras

### Structures
- pymatgen
- ASE

### Descriptors
- matminer

### Geometric deep learning
- PyTorch Geometric
- DGL

### Materials-oriented research frameworks
- matgl
- MEGNet-family models
- CHGNet-family approaches
- MACE and related atomistic ML methods

The frameworks are tools; students should understand the mathematical assumptions behind them before using pretrained models.


# 33. Exercise 4 — Classical ML versus deep learning

Using the same train/validation/test split:

1. Train linear regression.
2. Train a classical nonlinear model from Module 7.
3. Train the MLP.
4. Compare MAE, RMSE, and R².
5. Compare training time.
6. Compare model complexity.
7. Explain which model you would deploy and why.

A neural network should not be selected merely because it is more sophisticated.


# 34. Mini-project — Materials property prediction

Choose a property such as:

- elastic modulus
- yield strength
- hardness
- thermal conductivity
- formation energy
- band gap

Required:

1. scientific motivation
2. dataset description
3. descriptor definition
4. scientifically justified split
5. preprocessing
6. classical ML baseline
7. MLP
8. learning curves
9. hyperparameter experiment
10. regularization experiment
11. final test evaluation
12. residual analysis
13. physical interpretation
14. limitations


# 35. Mini-project — Microstructure image learning

Use real microscopy images or synthetic microstructures.

Possible objectives:

- phase classification
- porosity prediction
- grain-size prediction
- hardness prediction
- thermal-property prediction

Compare:

1. engineered image descriptors + classical ML
2. CNN

Discuss when learned representations are preferable to handcrafted descriptors.


# 36. Mini-project — Neural surrogate for numerical simulation

Generate training data using a finite-difference diffusion or heat-equation solver from the numerical-methods module.

Possible task:

`(D, t, x, boundary conditions) -> C(x,t)`

Then:

1. train a neural surrogate
2. compare with the numerical solver
3. quantify prediction error
4. test unseen parameter combinations
5. investigate extrapolation failure
6. compare computational cost


# 37. Capstone integration

The complete course can now form:

**materials physics → numerical model → simulation/data → feature engineering → classical ML → deep learning → screening/design**

A strong capstone may combine:

- NumPy/SciPy
- PDE/FDM simulation
- Pandas
- visualization
- materials descriptors
- classical ML
- neural networks
- uncertainty analysis
- physical validation


# 38. Assessment

### Mathematical

Explain:

- forward propagation
- activation functions
- loss functions
- gradient descent
- chain rule
- backpropagation
- regularization

### Computational

Implement:

- PyTorch datasets
- MLP regression
- classification
- CNN
- validation workflow
- early stopping

### Scientific

Evaluate:

- descriptor quality
- data leakage
- train/test strategy
- physical plausibility
- interpolation versus extrapolation
- model limitations


# 39. Final challenge

Answer:

> **Does deep learning provide a scientifically meaningful advantage over classical machine learning for your chosen materials problem?**

Your conclusion must discuss:

- data volume
- representation
- accuracy
- computational cost
- interpretability
- extrapolation
- physical constraints
- intended deployment

A strong conclusion can be that deep learning is **not** the best method. Scientific reasoning matters more than model complexity.


# 40. Key takeaways

1. A neural network is a parameterized mathematical function.
2. Backpropagation is repeated use of the chain rule.
3. Training minimizes a chosen objective; it does not prove physical correctness.
4. Validation is essential for model development.
5. The test set should remain untouched until final evaluation.
6. Regularization controls model complexity.
7. CNNs are useful for spatial/materials-image data.
8. Graph networks are natural for atomic structures.
9. Deep learning is most compelling when the representation contains information that handcrafted descriptors cannot easily capture.
10. Scientific validation and domain of applicability remain essential.

**Next direction:** scientific machine learning, graph neural networks, physics-informed neural networks, atomistic ML, and neural surrogate models.
